# Named Entity Recognition Project (Rule-Based)
# Ekstraksi Informasi Proposal Kegiatan EM UB

This notebook contains the implementation of a rule-based NER system for extracting information from activity proposals.

In [1]:
# 1. Install dependencies (run once in Codespaces)
%pip install PyMuPDF nltk Sastrawi pandas

Note: you may need to restart the kernel to use updated packages.


In [22]:
# 2. Import required libraries
import pandas as pd
import numpy as np
import json
import re
import ipywidgets as widgets
from IPython.display import display
import shutil
from utils.preprocess import clean_text
from utils.extract_pdf import extract_text_from_pdf

In [36]:
def extract_entities(text):
    entities = {
        "EVENT": [],
        "THEME": [],
        "ORG": [],
        "DEPARTMENT": [],
        "DATE_START": [],
        "DATE_END": [],
        "DURATION": [],
        "LOC": [],
        "PIC": [],
        "RESPONSIBLE": [],
        "BUDGET_PLAN": [],
        "FUND_SOURCE": [],
        "PARTICIPANT_COUNT": [],
        "PARTICIPANT_SCOPE": [],
        "PURPOSE": [],
        "BENEFIT": [],
        "OUTPUT": [],
        "CONTACT": []
    }

    # ==========================
    # 🧩 RULE-BASED NER SECTION
    # ==========================

    # EVENT / Nama Kegiatan
    match_event = re.findall(r"(?:(?:Kegiatan|Acara)\s*[:\-]?\s*)([A-Z][A-Za-z0-9\s&\-]+)", text, re.IGNORECASE)
    if match_event:
        entities["EVENT"].extend(match_event)

    # THEME / Tema Kegiatan
    match_theme = re.findall(r"(?:Tema\s*[:\-]?\s*)([\"“”A-Za-z0-9\s,.\-]+)", text, re.IGNORECASE)
    if match_theme:
        entities["THEME"].extend(match_theme)

    # ORG / Organisasi Penyelenggara
    match_org = re.findall(r"(Eksekutif\s*Mahasiswa|Badan\s*Eksekutif\s*Mahasiswa|Universitas\s+Brawijaya|Himpunan\s+Mahasiswa)", text, re.IGNORECASE)
    if match_org:
        entities["ORG"].extend(list(set(match_org)))

    # DEPARTMENT / Kementerian Pelaksana
    match_dep = re.findall(r"(Kementerian|Departemen)\s+[A-Z][A-Za-z\s]+", text)
    if match_dep:
        entities["DEPARTMENT"].extend(match_dep)

    # DATE_START dan DATE_END
    dates = re.findall(r"(\d{1,2}\s*(?:Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s*\d{4})", text, re.IGNORECASE)
    if len(dates) >= 1:
        entities["DATE_START"].append(dates[0])
    if len(dates) >= 2:
        entities["DATE_END"].append(dates[-1])

    # DURATION / Jadwal lengkap
    match_dur = re.findall(r"(\d{1,2}\s*(?:s\.d\.|\–|-|hingga|sampai)\s*\d{1,2}\s*(?:Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s*\d{4})", text, re.IGNORECASE)
    if match_dur:
        entities["DURATION"].extend(match_dur)

    # LOC / Lokasi kegiatan
    match_loc = re.findall(r"(?:bertempat|lokasi|di)\s+(Gedung|Aula|Lapangan|Ruang|Kampus|Universitas|Villa|Zoom)\s+[A-Za-z\s,]+", text, re.IGNORECASE)
    if match_loc:
        entities["LOC"].extend(list(set(match_loc)))

    # PIC / Ketua Pelaksana
    match_pic = re.findall(r"(?:Ketua\s*Pelaksana|Penanggung\s*Jawab)\s*[:\-]?\s*([A-Z][A-Za-z\s\.\']+)", text, re.IGNORECASE)
    if match_pic:
        entities["PIC"].extend(match_pic)

    # RESPONSIBLE / Penanggung Jawab umum
    match_resp = re.findall(r"(?:Penanggung\s*Jawab|Koordinator)\s*[:\-]?\s*([A-Z][A-Za-z\s\.\']+)", text, re.IGNORECASE)
    if match_resp:
        entities["RESPONSIBLE"].extend(match_resp)

    # BUDGET_PLAN / Anggaran Biaya
    match_budget = re.findall(r"Rp\s?[\d\.\,]+", text)
    if match_budget:
        entities["BUDGET_PLAN"].extend(match_budget)

    # FUND_SOURCE / Sumber Dana
    match_fund = re.findall(r"(?:Sumber\s*Dana|Pendanaan)\s*[:\-]?\s*([A-Za-z\s]+)", text, re.IGNORECASE)
    if match_fund:
        entities["FUND_SOURCE"].extend(match_fund)

    # PARTICIPANT_COUNT
    match_participant = re.findall(r"(\d{2,4})\s*(orang|peserta)", text, re.IGNORECASE)
    if match_participant:
        entities["PARTICIPANT_COUNT"].extend([" ".join(x) for x in match_participant])

    # PARTICIPANT_SCOPE
    match_scope = re.findall(r"(?:Peserta|Sasaran)\s*[:\-]?\s*([A-Za-z\s]+)", text, re.IGNORECASE)
    if match_scope:
        entities["PARTICIPANT_SCOPE"].extend(match_scope)

    # PURPOSE
    match_purpose = re.findall(r"(?:Tujuan|Maksud)\s*[:\-]?\s*([A-Za-z0-9\s,.\-]+)", text, re.IGNORECASE)
    if match_purpose:
        entities["PURPOSE"].extend(match_purpose)

    # BENEFIT
    match_benefit = re.findall(r"(?:Manfaat|Hasil)\s*[:\-]?\s*([A-Za-z0-9\s,.\-]+)", text, re.IGNORECASE)
    if match_benefit:
        entities["BENEFIT"].extend(match_benefit)

    # OUTPUT
    match_output = re.findall(r"(?:Luaran|Output)\s*[:\-]?\s*([A-Za-z0-9\s,.\-]+)", text, re.IGNORECASE)
    if match_output:
        entities["OUTPUT"].extend(match_output)

    # CONTACT
    match_contact = re.findall(r"(\+62|08)\d{8,13}", text)
    if match_contact:
        entities["CONTACT"].extend(match_contact)

    # Bersihkan entitas kosong
    entities = {k: v for k, v in entities.items() if v}

    return entities

In [37]:
# 3.1 Tampilkan tombol upload PDF
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(accept='.pdf', multiple=False)
display(upload)

FileUpload(value=(), accept='.pdf', description='Upload')

In [38]:
# 3.2 Simpan file yang sudah diupload
def save_uploaded_file(upload_widget):
    if not upload_widget.value:
        print("⚠️ Belum ada file yang diunggah. Jalankan ulang setelah memilih PDF.")
        return None

    # Format tuple/list (Jupyter, Codespaces)
    if isinstance(upload_widget.value, (tuple, list)):
        fileinfo = upload_widget.value[0]
        # Coba beberapa kemungkinan key nama file
        filename = (
            fileinfo.get("metadata", {}).get("name") or  # versi lama
            fileinfo.get("name") or                      # versi baru
            "uploaded.pdf"                               # fallback
        )
        with open(filename, "wb") as f:
            f.write(fileinfo["content"])
        return filename

    # Format dict (Colab lama)
    elif isinstance(upload_widget.value, dict):
        for filename, fileinfo in upload_widget.value.items():
            with open(filename, "wb") as f:
                f.write(fileinfo["content"])
            return filename

    else:
        raise TypeError(f"Tipe upload tidak dikenal: {type(upload_widget.value)}")

pdf_path = save_uploaded_file(upload)
if pdf_path:
    print("✅ File uploaded to:", pdf_path)

✅ File uploaded to: 003 - Proposal Training Organization.pdf


In [39]:
# 4. Process the uploaded PDF
if pdf_path:
    # Extract text from PDF
    raw_text = extract_text_from_pdf(pdf_path)
    print("Excerpt of extracted text:\n", raw_text[:500])

    # Preprocess text
    cleaned_text = clean_text(raw_text)
    print("\nText after preprocessing:\n", cleaned_text[:500])

    print("pdf_path =", pdf_path)
    print("cleaned_text exists?", 'cleaned_text' in locals())
    print("cleaned_text length:", len(cleaned_text) if 'cleaned_text' in locals() else None)
else:
    print("⚠️ Please upload a PDF file first")

Excerpt of extracted text:
  
 
 
003/EM/II/2023 
 
 
 
 
 
 
 
 
  
  
  
 
 
PROPOSAL KEGIATAN LKM 
“TRAINING ORGANIZATION” 
 
 
 
 
 
 
 
 
 
EKSEKUTIF MAHASISWA 
UNIVERSITAS BRAWIJAYA 
MALANG 
2023 
 
KEMENTERIAN PENDIDIKAN, KEBUDAYAAN, 
RISET, DAN TEKNOLOGI 
UNIVERSITAS BRAWIJAYA 
EKSEKUTIF MAHASISWA 
Gedung EM-DPM UB Lantai 1, Jalan Veteran 06C Malang, 65145  
Telp: 0895-2849-8557 Email: em@ub.ac.id 
 
 
 
ii 
HALAMAN PENGESAHAN 
 
1.  
Nama kegiatan 
: Training Organization 
2.  
Tempat kegiatan 
: Dalam jaringan (Z

Text after preprocessing:
 003 em ii 2023 proposal giat lkm training organization eksekutif mahasiswa universitas brawijaya malang 2023 menteri didik budaya riset teknologi universitas brawijaya eksekutif mahasiswa gedung em-dpm ub lantai 1 jalan veteran 06c malang 65145 telp 0895-2849-8557 email emub ac id ii halaman kesah 1 nama giat  training organization 2 giat  jaring zoom meeting jaring universitas brawijaya villa dewi batu 3 giat  selasa s d minggu 21 s d 26 

In [40]:
# 5. Run NER extraction
if pdf_path and 'cleaned_text' in locals():
    if 'extract_entities' in globals():
        # Jalankan ekstraksi entitas
        result = extract_entities(cleaned_text)

        # Simpan hasil ke JSON
        with open("output.json", "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print("\n=== EXTRACTION RESULTS (JSON) ===\n")
        print(json.dumps(result, indent=2, ensure_ascii=False))

        # ==============================
        # 🔹 Tampilkan hasil sebagai tabel
        # ==============================
        if result:
            df_result = pd.DataFrame(list(result.items()), columns=["Label", "Isi"])
            display(df_result)
        else:
            print("⚠️ Tidak ada entitas yang berhasil diekstraksi. Pastikan PDF berisi teks yang sesuai.")
    else:
        print("❌ Fungsi extract_entities belum didefinisikan. Jalankan cell definisinya terlebih dahulu.")
else:
    print("⚠️ Please complete the PDF upload and processing steps first")


=== EXTRACTION RESULTS (JSON) ===

{
  "EVENT": [
    "bentuk apresiasi semangat em ub 2023 susun acara lampir lampir 1 susun panitia lampir lampir 2 rencana anggar biaya lampir lampir 3 tutup proposal giat training organization aju tuju laksana giat capa tuju sasar giat hasil giat tentu partisipasi pihak itu dukung moril material menteri didik budaya riset teknologi universitas brawijaya eksekutif mahasiswa gedung em-dpm ub lantai 1 jalan veteran 06c malang 65145 telp 0895-2849-8557 email emub ac id 5 harap sukses giat ini panitia training organization terima kasih kerja sama dukung partisipasi bapak ibu ikan moga tuhan maha esa senantiasa limpah rahmat hidayah semua menteri didik budaya riset teknologi universitas brawijaya eksekutif mahasiswa gedung em-dpm ub lantai 1 jalan veteran 06c malang 65145 telp 0895-2849-8557 email emub ac id 6 lampir 1 susun acara senin s d jumat 20 s d 24 februari 2023 tanggal giat terang senin 20 februari 08 00-20 00 screening serta selasa 21 februari 0

,Label,Isi
0,EVENT,[bentuk apresiasi semangat em ub 2023 susun ac...
1,THEME,[giat 2 tuju giat 2 manfaat giat 3 tanggal l...
2,ORG,"[eksekutif mahasiswa, universitas brawijaya]"
3,DATE_START,[26 februari 2023]
4,DATE_END,[19 maret 2023]
5,DURATION,"[18-19 maret 2023, 18-19 maret 2023, 18-19 mar..."
6,PARTICIPANT_COUNT,"[400 orang, 400 orang]"
7,BENEFIT,[sama menteri didik budaya riset teknologi uni...
8,CONTACT,"[08, 08, 08]"
